In [1]:
import re
import os
import subprocess
import pandas as pd
import numpy as np
from pathlib import PurePath

from h_anonypy.modules_video import get_video_infomation, check_patient_ids, is_split_screen, check_split_screen
from h_anonypy.modules_video import get_video_metadata_ffprobe, get_video_metadata_opencv, capture_key_frames_by_video
from h_anonypy.modules_video import infer_capture_mode, assign_stereo_ch_names
import shutil


In [2]:
##### INPUT #####
# IMAGE_META : 전체 영상 메타정보 
# VIDEO_META : 전체 비디오 메타정보
# HUTOM_ID : 전체 hutom id 리스트
# video_dir : 반입 데이터 경로
# center: 반입 기관
# importdate: 반입 날짜
# organ: 조직 정보
# n_digits: hutom id 자리수

# split_str : 원본 파일명으로부터 새 파일명 정보 추출을 위한 파라미터
# select_n : 원본 파일명으로부터 새 파일명 정보 추출을 위한 파라미터

# IMAGE_META = pd.read_excel('./SHEET/IMAGE_META.xlsx', sheet_name=None)
# VIDEO_META = pd.read_excel('./SHEET/VIDEO_META.xlsx', sheet_name=None)
# HUTOM_ID = pd.read_excel('./SHEET/HUTOM_ID.xlsx', sheet_name=None)

base_dir = '/nas/nas6/DataTeam/'
video_dir = [base_dir+'COLON/[범부처전주기] 민병소교수님/211014_LAR']
center = ['범부처전주기']
importdate = ['20211014']
organ = ['COLON']
n_digits = 4

VIDEO_META = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/DB/VIDEO_META.xlsx')
hids_all = pd.read_excel('/disk1/users/da_cyh_0/PROJECT/DATA/anony/DB/HID_ALL.xlsx')


In [ ]:
## Start [ver.2025.08]
for i in range(len(video_dir)):
    # 1. 경로 내 비디오 파일 해시 추출
    source_info = get_video_infomation(video_dir[i])

    # 2. patient_id 수정 [여러 비디오가 존재하는 경우]
    posix = PurePath(video_dir[i])
    video_info = check_patient_ids(source_info, video_dir[i], id_path_index=len(posix.parts))
    video_info.insert(0, 'hutom_id', None)

    # 3. 메타정보 추출, 중복 제거 및 HUTOM ID 부여
    meta_all = VIDEO_META[organ[i]].copy()
    image_meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1

    save_dir = os.path.join(video_dir[i],'capture') 
    video_info[['size(bytes)','width','height','codec_name','fps','nb_frames','duration']] = None
    ids = video_info['patient_id'].unique().tolist()
    for j in range(len(ids)):
        # 3.1 메타 정보 추출
        
        check_sample = video_info[video_info['patient_id'] == ids[j]]
        check_idx = check_sample.index.tolist()

        idx_ch = 0
        idx_cont = 1  
        idx_split = 1 
        for k in range(len(check_idx)):
            filepath = check_sample.loc[check_idx[k],'filepath']
            meta_ffprobe = get_video_metadata_ffprobe(filepath)
            if not meta_ffprobe: 
                video_info.loc[check_idx[k],'format'] = 'dameged_file'
                continue 
            meta_opencv = get_video_metadata_opencv(filepath)
            video_stream = meta_ffprobe['streams'][0]
            video_info.loc[check_idx[k],'size(bytes)'] = os.path.getsize(filepath)
            video_info.loc[check_idx[k],'width'] = video_stream.get('width', meta_opencv['width'])
            video_info.loc[check_idx[k],'height'] = video_stream.get('height', meta_opencv['height'])
            video_info.loc[check_idx[k],'codec_name'] = video_stream.get('codec_name')
            try:
                video_info.loc[check_idx[k],'fps'] = eval(video_stream.get('avg_frame_rate'))
            except:
                video_info.loc[check_idx[k],'fps'] = meta_opencv['fps']
            video_info.loc[check_idx[k],'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
            video_info.loc[check_idx[k],'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

            # 캡처 이미지 및 채널명 생성
            frames = capture_key_frames_by_video(filepath, video_dir[i], save_dir)
            # check split image
            name, ext = os.path.splitext(os.path.basename(filepath))
            check_vertical, check_horizontal = is_split_screen(frames[1])
            if len(check_vertical) == 1 and len(check_horizontal) == 1:
                if 'ch' in name:
                    channel = re.search(r"(ch\d+)", name).group(1)
                    if idx_cont > 1:
                        if reset_channel != channel:
                            idx_cont = 1
                    ch_name = f"{channel}_{idx_cont:02d}"
                else:
                    ch_name = f"ch{idx_ch}_{idx_cont:02d}" # ch0_01
                idx_cont +=1
                reset_channel = channel
            else:
                ch_name = f"split_{idx_split:02d}" # split_01, split_02
                idx_split += 1
            video_info.loc[check_idx[k],'ch_name'] = ch_name
            
        # 3.2. ID 재구성 > 확인이 되는 경우 재추출
        if split_str is not None:
            id_re = ids[j].split(split_str)[select_n] 
            video_info.loc[check_idx, "patient_id"] = id_re
        
        # 3.3. 휴톰 아이디 생성
        hash_list = check_sample['hash'].unique().tolist()
        id_list = check_sample['patient_id'].unique().tolist()
        dup = meta_all[meta_all['Hash'].isin(hash_list)]
        dup_img = image_meta_all[image_meta_all['PatientID'].isin(id_list)]

        if len(dup)+len(dup_img) == 0:
            hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
            video_info.loc[check_idx, "hutom_id"] = hutomid
            id_number += 1
        elif len(dup)+len(dup_img) > 0:
            if len(dup) == 0:
                hutomid = dup_img["hutom_id"].tolist()[0]
            elif len(dup_img) == 0:
                hutomid = dup["hutom_id"].tolist()[0]
            video_info.loc[check_idx, "hutom_id"] = hutomid

    # 4. 익명화
    for l in range(len(video_info)):

        filepath = video_info.loc[l,'filepath']

        hutomid = video_info.loc[l,'hutom_id']
        anonyid = f"{hutomid}_{video_info.loc[l,'ch_name']}.{video_info.loc[l,'format']}"
        anony_folder = os.path.join(video_dir[i], 'ANONYMOUS', hutomid)
        anony_filename = os.path.join(anony_folder, anonyid)

        os.makedirs(anony_folder, exist_ok=True)

        if "ch" in video_info.loc[l,'ch_name']:
            anony_filename = os.path.join(video_dir[i], 'ANONYMOUS', hutomid, anonyid)
            shutil.copyfile(filepath, anony_filename)

            video_info.loc[l,'filepath'] = str(PurePath(*PurePath(filepath).parts[2:]))
            video_info.loc[l,'anony_filepath'] = str(PurePath(*PurePath(anony_filename).parts[2:]))

    # 5. Add information > check !!!
    ids_add = pd.DataFrame(video_info['hutom_id'].unique().tolist(),
                           columns=['hutom_id'])
    ids_add['video'] = 'O'
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["video"] = ids_all_add["video_x"].fillna(ids_all_add["video_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]

    video_info = video_info.rename(columns={'filepath':'RAW_PATH',
                                            'anony_filepath':'SOURCE_PATH',
                                            'hash':'Hash'})
    meta_all_add = pd.concat([meta_all, 
                              video_info[meta_all.columns.tolist()]], ignore_index=True)

##### OUTPUT > VIDEO_META, HUTOM_ID 업데이트 및 저장
# VIDEO_META[organ[i]]
# meta_all_add



In [ ]:
## Start [ver.2025.11]

i=0

source_info = get_video_infomation(video_dir[i])
posix = PurePath(video_dir[i])
video_info = check_patient_ids(source_info, video_dir[i], id_path_index=len(posix.parts))
video_info.insert(0, 'hutom_id', None)

# 3. 메타정보 추출, 중복 제거 및 HUTOM ID 부여
if organ[i] in list(HUTOM_ID.keys()):
    meta_all = VIDEO_META[organ[i]].copy()
    image_meta_all = IMAGE_META[organ[i]].copy()
    ids_all = HUTOM_ID[organ[i]].copy()
    mask = ~ids_all['hutom_id'].astype(str).str.contains('FDA', case=False, na=False)
    hutom_ids = ids_all.loc[mask,'hutom_id'].dropna().unique().tolist()
    nums = [int(m.group(1)) for s in hutom_ids if (m := re.search(r'(\d+)$', s))]
    id_number = np.sort(nums)[-1] + 1
else:
    id_number = 1

save_dir = os.path.join(video_dir[i],'capture') 
video_info[['size(bytes)','width','height','codec_name','fps','nb_frames','duration','split_info','capture_mode']] = None
ids = video_info['patient_id'].unique().tolist()
for j in range(len(ids)):
    # 3.1 메타 정보 추출
    
    check_sample = video_info[video_info['patient_id'] == ids[j]]
    check_idx = check_sample.index.tolist()

    capture_mode = infer_capture_mode(check_sample['filepath'].tolist())
    video_info.loc[check_idx, 'capture_mode'] = capture_mode
    for k in range(len(check_idx)):
        filepath = check_sample.loc[check_idx[k],'filepath']
        meta_ffprobe = get_video_metadata_ffprobe(filepath)
        if not meta_ffprobe: 
            video_info.loc[check_idx[k],'format'] = 'dameged_file'
            continue 
        meta_opencv = get_video_metadata_opencv(filepath)
        video_stream = meta_ffprobe['streams'][0]
        video_info.loc[check_idx[k],'size(bytes)'] = os.path.getsize(filepath)
        video_info.loc[check_idx[k],'width'] = video_stream.get('width', meta_opencv['width'])
        video_info.loc[check_idx[k],'height'] = video_stream.get('height', meta_opencv['height'])
        video_info.loc[check_idx[k],'codec_name'] = video_stream.get('codec_name')
        try:
            video_info.loc[check_idx[k],'fps'] = eval(video_stream.get('avg_frame_rate'))
        except:
            video_info.loc[check_idx[k],'fps'] = meta_opencv['fps']
        video_info.loc[check_idx[k],'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
        video_info.loc[check_idx[k],'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

        # 3.2 캡처 이미지 생성
        frames = capture_key_frames_by_video(filepath, video_dir[i], save_dir)
        video_info.loc[check_idx[k],'split_info'] = 'capture_pairing'
        video_info.at[check_idx[k], 'frames'] = frames

    ch_name_map = assign_stereo_ch_names(video_info.loc[check_idx])
    for row_index, ch_name in ch_name_map.items():
        video_info.loc[row_index, 'ch_name'] = ch_name

    # 3.3. 휴톰 아이디 생성 [중복 체크 후 아이디 생성]
    hash_list = check_sample['hash'].unique().tolist()
    id_list = check_sample['patient_id'].unique().tolist()
    if organ[i] in list(HUTOM_ID.keys()):
        dup = meta_all[meta_all['Hash'].isin(hash_list)]
        dup_img = image_meta_all[image_meta_all['PatientID'].isin(id_list)]
    else:
        dup, dup_img = [], []
    
    if len(dup)+len(dup_img) == 0:
        hutomid = f"{organ[i]}{id_number:0{n_digits}d}"
        video_info.loc[check_idx, "hutom_id"] = hutomid
        id_number += 1
    elif len(dup)+len(dup_img) > 0:
        if len(dup) == 0:
            hutomid = dup_img["hutom_id"].tolist()[0]
        elif len(dup_img) == 0:
            hutomid = dup["hutom_id"].tolist()[0]
        video_info.loc[check_idx, "hutom_id"] = hutomid

# 4. 익명화
filepath = None
anony_filename = None
cmd = [
    "ffmpeg",
    "-i", filepath,
    "-c:v", "libx264",
    "-profile:v", "high",
    "-level", "4.0",
    "-pix_fmt", "yuv420p",
    "-s", "1280x1024",
    "-r", "30",
    "-b:v", "6962k",
    "-c:a", "aac",
    "-profile:a", "aac_low",
    "-ar", "48000",
    "-ac", "2",
    "-b:a", "128k",
    "-movflags", "+faststart",
    anony_filename
]
jobs = []
for l in range(len(video_info)):
    
    filepath = video_info.loc[l,'filepath']
    hutomid = video_info.loc[l,'hutom_id']
    anonyid = f"{hutomid}_{video_info.loc[l,'ch_name']}.mp4"
    anony_folder = os.path.join(video_dir[i], 'ANONYMOUS', hutomid)
    os.makedirs(anony_folder, exist_ok=True)
    anony_filename = os.path.join(video_dir[i], 'ANONYMOUS', hutomid, anonyid)

    run_cmd = cmd.copy()
    run_cmd[2] = filepath
    run_cmd[-1] = anony_filename
    jobs.append((l, filepath, anony_filename, run_cmd))

    video_info.loc[l,'filepath'] = str(PurePath(*PurePath(filepath).parts[2:]))
    video_info.loc[l,'anony_filepath'] = str(PurePath(*PurePath(anony_filename).parts[2:]))

video_info = video_info.rename(columns={'filepath':'RAW_PATH',
                                        'anony_filepath':'SOURCE_PATH',
                                        'hash':'Hash'})
video_info['Center'] = center[i]
video_info['ImportDate'] = importdate[i]
video_info.to_excel(video_dir[i] + '/video_info_' + importdate[i] + '.xlsx', index=None)

for _, _, _, run_cmd in jobs:
    subprocess.run(run_cmd, check=True)

# 3.4. 추출 메타 정보 저장

video_info



In [ ]:
# 5. Add information > check !!!
ids_add = pd.DataFrame(video_info['hutom_id'].unique().tolist(),
                        columns=['hutom_id'])
ids_add['video'] = 'O'

if organ[i] in list(HUTOM_ID.keys()):
    if 'capture_mode' not in meta_all.columns:
        meta_all['capture_mode'] = None
    ids_all_add = pd.merge(ids_all, ids_add, on='hutom_id', how='outer')
    ids_all_add["video"] = ids_all_add["video_x"].fillna(ids_all_add["video_y"])
    ids_all_add = ids_all_add[['hutom_id','dicom','video']]
    meta_all_add = pd.concat([meta_all, video_info[meta_all.columns.tolist()]], ignore_index=True)
else:
    ids_all_add = ids_add.copy()
    ids_all_add.insert(1,'dicom', None)
    col_list = ['hutom_id', 'Hash', 'RAW_PATH', 'SOURCE_PATH',
            'size(bytes)', 'width', 'height', 'codec_name', 'fps', 'nb_frames',
            'duration', 'split_info', 'capture_mode', 'ch_name', 'ImportDate']
    meta_all_add = video_info[col_list].copy()

VIDEO_META[organ[i]] = meta_all_add
with pd.ExcelWriter('./SHEET/VIDEO_META.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in VIDEO_META.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)
        
HUTOM_ID[organ[i]] = ids_all_add
with pd.ExcelWriter('./SHEET/HUTOM_ID.xlsx', engine='openpyxl') as writer:
    for sheet_name, data in HUTOM_ID.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)


In [3]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text

def build_queries(qualified: str):
    """
    입력: 'schema.table' 형식의 문자열
    출력: (count_sql, select_sql)
    """
    qualified = qualified.strip().strip('"').strip("'")
    if '.' in qualified:
        schema, table = qualified.split('.', 1)
    else:
        schema, table = 'public', qualified  # 스키마 생략 시 public 가정

    # 1) 비인용(그대로) 버전
    count_sql  = f"SELECT COUNT(*) AS n FROM {schema}.{table}"

    # 2) 식별자 인용(안전) 버전
    select_sql = f'SELECT * FROM "{schema}"."{table}"'

    return count_sql, select_sql

def extract_db(engine, table_name):
    
    count_sql, select_sql = build_queries(table_name)

    # 크기 확인
    n = pd.read_sql(count_sql, engine)["n"][0]
    print("rows:", n)
    
    # Load DB
    df_head = pd.read_sql(text(select_sql), engine)
    
    return df_head

USER = 'postgres'
PWD  = 'postgres'
HOST = '192.168.16.16'      # 또는 DB 서버 주소
PORT = 6543
DB   = 'Hutom'

URL = f'postgresql+psycopg2://{USER}:{PWD}@{HOST}:{PORT}/{DB}'
engine = create_engine(URL, pool_pre_ping=True)

# query = """
# SELECT table_name
# FROM information_schema.tables
# WHERE table_schema = 'hutom_bronze'
# ORDER BY table_name;
# """
# df_tables = pd.read_sql(query, engine)
# df_tables

# VIDEO_META = pd.read_excel('DATA/anony/SHEET/VIDEO_META.xlsx', sheet_name=None)
VIDEO_META = extract_db(engine, 'public.tbl_data_import_video')

hids_all = extract_db(engine, 'hutom_bronze.tbl_id_token_linkage')
HUTOM_ID = hids_all[hids_all['hutom_id'].astype(str).str.contains(organ[0], na=False)]
HUTOM_ID = HUTOM_ID[~HUTOM_ID['hutom_id'].astype(str).str.contains('FDA', na=False)]
HUTOM_ID = HUTOM_ID.sort_values('hutom_id')
hids = HUTOM_ID['hutom_id'].tolist()
nums = [int(m.group(1)) for s in hids if (m := re.search(r'(\d+)$', s))]
id_number = np.sort(nums)[-1] + 1




rows: 22284
rows: 23291


In [7]:
os.path.join(video_dir[i], 'VIDEO_MATA_'+organ[i]+'_'+importdate[i]+'.xlsx')


'/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/211014_LAR/VIDEO_MATA_COLON_20211014.xlsx'

In [3]:
# 원본 데이터 기준, 기본 정보 추출

i = 0 
save_dir = os.path.join(video_dir[i],'capture') 

HUTOM_ID = hids_all[hids_all['hutom_id'].astype(str).str.contains(organ[i], na=False)]
HUTOM_ID = HUTOM_ID[~HUTOM_ID['hutom_id'].astype(str).str.contains('FDA', na=False)]
HUTOM_ID = HUTOM_ID.sort_values('hutom_id')
hids = HUTOM_ID['hutom_id'].tolist()
nums = [int(m.group(1)) for s in hids if (m := re.search(r'(\d+)$', s))]
id_number = np.sort(nums)[-1] + 1

source_info = get_video_infomation(video_dir[i])
posix = PurePath(video_dir[i])
video_info = check_patient_ids(source_info, base_dir, id_path_index=len(posix.parts))
video_info.insert(0, 'hutom_id', None)

video_info[['new_hash','size(bytes)','width','height','codec_name','fps','nb_frames','duration','split','data_type']] = None
video_info['data_type'] = organ[i]
ids = video_info['patient_id'].unique().tolist()

video_info


,hutom_id,patient_id,patient_name,filename,filefolder,filepath,hash,format,new_hash,size(bytes),width,height,codec_name,fps,nb_frames,duration,split,data_type
0,None,10275867_09032021_132146,10275867_09032021_132146,ch1_video_01.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,f7997fa3f87cc86bfdf8d1e3836eeaf6fecd042f9f2ac7...,mp4,None,None,None,None,None,None,None,None,None,COLON
1,None,10275867_09032021_132146,10275867_09032021_132146,ch2_video_02.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d6493158fcc296ad33c91e221e3db4cf37c67eba6d5eb3...,mp4,None,None,None,None,None,None,None,None,None,COLON
2,None,10338060_07262021_065513,10338060_07262021_065513,ch1_video_01.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,60b5d84b0bc99e54bd9b546b3b7392e65c165d86da1fa8...,mp4,None,None,None,None,None,None,None,None,None,COLON
3,None,10338060_07262021_065513,10338060_07262021_065513,ch1_video_03.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,b35c5bddec612d91bcbdccaf2779bef7266f0716cca3d7...,mp4,None,None,None,None,None,None,None,None,None,COLON
4,None,10338060_07262021_065513,10338060_07262021_065513,ch2_video_02.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,5f78dac84e2503270accc337d709ac450b2d368f0b7dd1...,mp4,None,None,None,None,None,None,None,None,None,COLON
5,None,10338060_07262021_065513,10338060_07262021_065513,ch2_video_04.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,699571f366ee617d4dc795690015ae991080bfd566e036...,mp4,None,None,None,None,None,None,None,None,None,COLON
6,None,10344161_08132021_203806,10344161_08132021_203806,ch1_video_01.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d78dfbaa2e8d18c2bea631a5e1c6b37c6bdc4e13cf2b36...,mp4,None,None,None,None,None,None,None,None,None,COLON
7,None,10344161_08132021_203806,10344161_08132021_203806,ch2_video_02.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,1ece927367169917d7fe5839e27c27d5caf1af9301cc03...,mp4,None,None,None,None,None,None,None,None,None,COLON
8,None,10356555_08182021_171016,10356555_08182021_171016,ch1_video_01.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d7223bb016c87e1acf36daac34fcb9f44e6e33fb125367...,mp4,None,None,None,None,None,None,None,None,None,COLON
9,None,10356555_08182021_171016,10356555_08182021_171016,ch2_video_02.mp4,/Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,7871859f665b4d4dbd863f257353f7d109a6252e3910e4...,mp4,None,None,None,None,None,None,None,None,None,COLON


In [4]:
# 메타 정보 추출
for j in range(len(ids)):
    
    check_sample = video_info[video_info['patient_id'] == ids[j]]
    check_idx = check_sample.index.tolist()

    capture_mode = infer_capture_mode(check_sample['filepath'].tolist())
    video_info.loc[check_idx, 'mode'] = capture_mode
    for k in range(len(check_idx)):
        filepath = check_sample.loc[check_idx[k],'filepath']
        # 메타 정보 추출
        meta_ffprobe = get_video_metadata_ffprobe(filepath)
        if not meta_ffprobe: 
            video_info.loc[check_idx[k],'format'] = 'dameged_file'
            continue 
        meta_opencv = get_video_metadata_opencv(filepath)
        video_stream = meta_ffprobe['streams'][0]
        video_info.loc[check_idx[k],'size(bytes)'] = os.path.getsize(filepath)
        video_info.loc[check_idx[k],'width'] = video_stream.get('width', meta_opencv['width'])
        video_info.loc[check_idx[k],'height'] = video_stream.get('height', meta_opencv['height'])
        video_info.loc[check_idx[k],'codec_name'] = video_stream.get('codec_name')
        try:
            video_info.loc[check_idx[k],'fps'] = eval(video_stream.get('avg_frame_rate'))
        except:
            video_info.loc[check_idx[k],'fps'] = meta_opencv['fps']
        video_info.loc[check_idx[k],'nb_frames'] = video_stream.get('nb_frames', meta_opencv['nb_frames'])
        video_info.loc[check_idx[k],'duration'] = float(video_stream.get('duration', meta_opencv['duration']))

        # 캡처 이미지 생성 및 stereo pair 확인용 정보 수집
        frames = capture_key_frames_by_video(filepath, video_dir[i], save_dir)
        check_screen = check_split_screen(frames)
        video_info.loc[check_idx[k],'split'] = check_screen

    # 채널명 할당
    ch_name_map = assign_stereo_ch_names(video_info.loc[check_idx])
    for row_index, ch_name in ch_name_map.items():
        video_info.loc[row_index, 'ch_name'] = ch_name

    # ID 재구성 > 확인이 되는 경우 재추출
    check_hashs = VIDEO_META[VIDEO_META['hash'].isin(video_info.loc[check_idx, 'hash'].tolist())]
    if len(check_hashs) == 0:
        hutomid = f"{organ[i]}{id_number:0{4}d}"
        id_number += 1
    else:
        hutomid = check_hashs["hutom_id"].tolist()[0]
    video_info.loc[check_idx, "hutom_id"] = hutomid


In [ ]:
# 익명화를 위한 비디오 변환 및 추가 정보 추출
cmd = [
    "ffmpeg",
    "-i", None,
    "-c:v", "libx264",
    "-profile:v", "high",
    "-level", "4.0",
    "-pix_fmt", "yuv420p",
    "-s", "1280x1024",
    "-r", "30",
    "-b:v", "6962k",
    "-c:a", "aac",
    "-profile:a", "aac_low",
    "-ar", "48000",
    "-ac", "2",
    "-b:a", "128k",
    "-movflags", "+faststart",
    None
]
cmd_list = []
for l in range(len(video_info)):

    filepath = video_info.loc[l,'filepath']
    cmd[2] = filepath

    hutomid = video_info.loc[l,'hutom_id']
    anonyid = f"{video_info.loc[l,'ch_name']}.mp4"
    anony_folder = os.path.join(video_dir[i], 'ANONYMOUS', hutomid)
    os.makedirs(anony_folder, exist_ok=True)

    # if "ch" in video_info.loc[l,'ch_name']:
    anony_filename = os.path.join(video_dir[i], 'ANONYMOUS', hutomid, anonyid)
    cmd[-1] = anony_filename
    cmd_list.append(cmd)

    video_info.loc[l,'source_filepath'] = os.path.join('/Volumes/SourceData', organ[i], 'VIDEO', hutomid, anonyid)
    video_info.loc[l,'source_filename'] = anonyid

video_info


,hutom_id,patient_id,patient_name,filename,filefolder,filepath,hash,format,new_hash,size(bytes),...,codec_name,fps,nb_frames,duration,split,data_type,mode,ch_name,source_filepath,source_filename
0,COLON0049,10275867_09032021_132146,10275867_09032021_132146,ch1_video_01.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10275...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,f7997fa3f87cc86bfdf8d1e3836eeaf6fecd042f9f2ac7...,mp4,None,15114618741,...,h264,29.999942,520819,17360.666667,None,COLON,stereo,ch1_01,/Volumes/SourceData/COLON/VIDEO/COLON0049/ch1_...,ch1_01.mp4
1,COLON0049,10275867_09032021_132146,10275867_09032021_132146,ch2_video_02.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10275...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d6493158fcc296ad33c91e221e3db4cf37c67eba6d5eb3...,mp4,None,15113212321,...,h264,30.0,520817,17360.566667,None,COLON,stereo,ch2_01,/Volumes/SourceData/COLON/VIDEO/COLON0049/ch2_...,ch2_01.mp4
2,COLON0050,10338060_07262021_065513,10338060_07262021_065513,ch1_video_01.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10338...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,60b5d84b0bc99e54bd9b546b3b7392e65c165d86da1fa8...,mp4,None,17891063540,...,h264,29.999903,616540,20551.4,None,COLON,stereo,ch1_01,/Volumes/SourceData/COLON/VIDEO/COLON0050/ch1_...,ch1_01.mp4
3,COLON0050,10338060_07262021_065513,10338060_07262021_065513,ch1_video_03.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10338...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,b35c5bddec612d91bcbdccaf2779bef7266f0716cca3d7...,mp4,None,2700675762,...,h264,30.0,92991,3099.7,None,COLON,stereo,ch1_02,/Volumes/SourceData/COLON/VIDEO/COLON0050/ch1_...,ch1_02.mp4
4,COLON0050,10338060_07262021_065513,10338060_07262021_065513,ch2_video_02.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10338...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,5f78dac84e2503270accc337d709ac450b2d368f0b7dd1...,mp4,None,17891112809,...,h264,29.999951,616538,20551.3,None,COLON,stereo,ch2_01,/Volumes/SourceData/COLON/VIDEO/COLON0050/ch2_...,ch2_01.mp4
5,COLON0050,10338060_07262021_065513,10338060_07262021_065513,ch2_video_04.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10338...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,699571f366ee617d4dc795690015ae991080bfd566e036...,mp4,None,2701503102,...,h264,30.0,92991,3099.7,None,COLON,stereo,ch2_02,/Volumes/SourceData/COLON/VIDEO/COLON0050/ch2_...,ch2_02.mp4
6,COLON0051,10344161_08132021_203806,10344161_08132021_203806,ch1_video_01.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10344...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d78dfbaa2e8d18c2bea631a5e1c6b37c6bdc4e13cf2b36...,mp4,None,12816807202,...,h264,30.0,441776,14725.866667,None,COLON,stereo,ch1_01,/Volumes/SourceData/COLON/VIDEO/COLON0051/ch1_...,ch1_01.mp4
7,COLON0051,10344161_08132021_203806,10344161_08132021_203806,ch2_video_02.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10344...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,1ece927367169917d7fe5839e27c27d5caf1af9301cc03...,mp4,None,12816695842,...,h264,30.0,441776,14725.866667,None,COLON,stereo,ch2_01,/Volumes/SourceData/COLON/VIDEO/COLON0051/ch2_...,ch2_01.mp4
8,COLON0052,10356555_08182021_171016,10356555_08182021_171016,ch1_video_01.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10356...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,d7223bb016c87e1acf36daac34fcb9f44e6e33fb125367...,mp4,None,10914964402,...,h264,30.0,375507,12516.9,None,COLON,stereo,ch1_01,/Volumes/SourceData/COLON/VIDEO/COLON0052/ch1_...,ch1_01.mp4
9,COLON0052,10356555_08182021_171016,10356555_08182021_171016,ch2_video_02.mp4,RawData/COLON/[범부처전주기] 민병소교수님/211014_LAR/10356...,/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...,7871859f665b4d4dbd863f257353f7d109a6252e3910e4...,mp4,None,10915539213,...,h264,29.99992,375506,12516.9,None,COLON,stereo,ch2_01,/Volumes/SourceData/COLON/VIDEO/COLON0052/ch2_...,ch2_01.mp4


In [ ]:
recodec = True
for command in cmd_list:
    
    if recodec == True:
        subprocess.run(command, check=True)
    else:
        shutil.copyfile(command[2], command[-1])


('/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/211014_LAR/LIM_10344874_09082021_230726/ch2_video_02.mp4',
 '/nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/211014_LAR/ANONYMOUS/COLON0082/ch2_01.mp4')

In [9]:
display(video_info.loc[0], VIDEO_META.loc[0])


hutom_id                                                COLON0049
patient_id                               10275867_09032021_132146
patient_name                             10275867_09032021_132146
filename                                         ch1_video_01.mp4
filefolder      /Volumes/RawData/COLON/[범부처전주기] 민병소교수님/211014_...
filepath        /nas/nas6/DataTeam/COLON/[범부처전주기] 민병소교수님/21101...
hash            f7997fa3f87cc86bfdf8d1e3836eeaf6fecd042f9f2ac7...
format                                                        mp4
new_hash                                                     None
size(bytes)                                           15114618741
width                                                        1280
height                                                       1024
codec_name                                                   h264
fps                                                     29.999942
nb_frames                                                  520819
duration  

Unnamed: 0                                                             0
video_file_id                                                      23387
data_type                                                             GB
rawdata_path           RawData/GB/[VIDEO]신촌세브란스병원-김성현_250722_NN건/AI c...
rawdata_filename                                              CMC_06.mp4
sourcedata_path        /Volumes/SourceData/GB/VIDEO/GB0664/GB0664_ch0...
sourcedata_filename                         GB0664_ch0_01_refined_v2.mp4
request_id                                                           114
hutom_id                                                          GB0664
patient_id                                                           NaN
size                                                          1338097780
hash                   c80ff3617e98d0383353ee36c1b2add35331cf11790065...
width                                                               1280
height                                             